<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/EmotionTracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 20.7 MB/s eta 0:00:00


In [15]:
"""
ArvyaX — Neural Network with Multi-Input Architecture
=======================================================

Architecture overview:
  INPUT 1 → journal_text        → TF-IDF (300) → Dense branch A
  INPUT 2 → stress_level        → numeric       ─┐
  INPUT 3 → face_emotion_hint   → Embedding(8)  ─┤→ Dense branch B (relationship branch)
  INPUT 4 → reflection_quality  → Embedding(4)  ─┘
  INPUT 5 → other numeric/ord   → Dense branch C
  INPUT 6 → previous_day_mood   → Embedding(8)
  INPUT 7 → ambience_type       → Embedding(8)

  Branch A + Branch B (relationship) + Branch C + mood + amb
        → Shared hidden layers
        → HEAD 1 → emotional_state  (Softmax Classification)
        → HEAD 2 → intensity        (Linear Regression)

  Recommendations generated from (predicted_state, intensity,
  stress_level, energy_level, time_of_day)
"""

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from gensim.models import Word2Vec

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, mean_absolute_error)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)
os.makedirs('outputs', exist_ok=True)

print("=" * 65)
print("  ArvyaX — NN  |  TF-IDF + Embeddings + Relationship Modeling")
print("=" * 65)

# ══════════════════════════════════════════════════════════
# LOAD
# ══════════════════════════════════════════════════════════
train_raw = pd.read_csv('Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')
test_raw  = pd.read_csv('arvyax_test_inputs_120.xlsx - Sheet1.csv')
print(f"\n  Train: {len(train_raw)}  |  Test: {len(test_raw)}")

# ══════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════
DROP_COLS    = {'id'}
NUMERIC_COLS = ['duration_min', 'sleep_hours', 'energy_level', 'stress_level']
ORDINAL_COLS = ['time_of_day']
CAT_COLS     = ['previous_day_mood', 'face_emotion_hint', 'ambience_type']
TEXT_COL     = 'journal_text'
EMBED_DIM    = 8
TFIDF_DIM    = 300

# ══════════════════════════════════════════════════════════
# PREPROCESSING
# ══════════════════════════════════════════════════════════
print("\n[1] Preprocessing ...")

for df in [train_raw, test_raw]:
    for c in DROP_COLS:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)

# time_of_day → ordinal
TIME_MAP = {'morning':4,'earlymorning':4,'early_morning':4,
            'afternoon':2,'evening':3,'night':1}
def map_time(v):
    return TIME_MAP.get(str(v).strip().lower().replace(' ',''), 2)

train_raw['time_of_day'] = train_raw['time_of_day'].apply(map_time)
test_raw['time_of_day']  = test_raw['time_of_day'].apply(map_time)

# Numeric NaN → train median
print("  Numeric medians:")
for col in NUMERIC_COLS:
    train_raw[col] = pd.to_numeric(train_raw[col], errors='coerce')
    test_raw[col]  = pd.to_numeric(test_raw[col],  errors='coerce')
    med = train_raw[col].median()
    train_raw[col] = train_raw[col].fillna(med)
    test_raw[col]  = test_raw[col].fillna(med)
    print(f"    {col:15s} = {med:.2f}")

# stress_level → scale 0-1 for relationship branch
stress_max = train_raw['stress_level'].max()
train_raw['stress_norm'] = train_raw['stress_level'] / stress_max
test_raw['stress_norm']  = test_raw['stress_level']  / stress_max

# Categorical NaN → 'neutral'
for col in CAT_COLS:
    train_raw[col] = train_raw[col].fillna('neutral').astype(str).str.strip().str.lower()
    test_raw[col]  = test_raw[col].fillna('neutral').astype(str).str.strip().str.lower()

# reflection_quality → fill + lowercase
if 'reflection_quality' in train_raw.columns:
    train_raw['reflection_quality'] = train_raw['reflection_quality'].fillna('unknown').astype(str).str.strip().str.lower()
# test may not have reflection_quality
if 'reflection_quality' not in test_raw.columns:
    test_raw['reflection_quality'] = 'unknown'
else:
    test_raw['reflection_quality']  = test_raw['reflection_quality'].fillna('unknown').astype(str).str.strip().str.lower()

# Text NaN → 'no reflection'
train_raw[TEXT_COL] = train_raw[TEXT_COL].fillna('no reflection').astype(str)
test_raw[TEXT_COL]  = test_raw[TEXT_COL].fillna('no reflection').astype(str)

# Merge calm+neutral (reduces confusion between near-identical classes)
MERGE_MAP = {'calm': 'calm_neutral', 'neutral': 'calm_neutral'}
train_raw['emotional_state'] = train_raw['emotional_state'].map(
    lambda x: MERGE_MAP.get(str(x).strip().lower(), str(x).strip().lower()))

print(f"\n  Classes after calm+neutral merge:")
for cls, cnt in train_raw['emotional_state'].value_counts().items():
    print(f"    {cls:15s} {cnt:3d} ({cnt/len(train_raw)*100:.1f}%)")

# ══════════════════════════════════════════════════════════
# WORD2VEC — categorical + journal text corpus
# ══════════════════════════════════════════════════════════
print("\n[2] Training Word2Vec ...")

def tokenize(value):
    return str(value).strip().lower().replace(' ','_').split('_')

corpus = []
for col in CAT_COLS + ['reflection_quality']:
    for val in pd.concat([train_raw[col], test_raw[col]]).unique():
        corpus.append(tokenize(val))
for text in train_raw[TEXT_COL]:
    tokens = str(text).lower().split()
    if tokens:
        corpus.append(tokens)

w2v = Word2Vec(sentences=corpus, vector_size=EMBED_DIM,
               window=3, min_count=1, workers=4, seed=42, epochs=100)

def get_emb(value):
    tokens = tokenize(value)
    vecs = [w2v.wv[t] if t in w2v.wv else np.zeros(EMBED_DIM) for t in tokens]
    return np.mean(vecs, axis=0)

def embed_col(series):
    return np.vstack([get_emb(v) for v in series]).astype(np.float32)

print(f"  Vocabulary size: {len(w2v.wv)}")

# ══════════════════════════════════════════════════════════
# TF-IDF on journal_text
# ══════════════════════════════════════════════════════════
print("\n[3] TF-IDF on journal_text ...")

tfidf    = TfidfVectorizer(max_features=TFIDF_DIM, ngram_range=(1,2),
                            sublinear_tf=True, min_df=2, lowercase=True)
tfidf_tr = tfidf.fit_transform(train_raw[TEXT_COL]).toarray().astype(np.float32)
tfidf_te = tfidf.transform(test_raw[TEXT_COL]).toarray().astype(np.float32)
print(f"  TF-IDF shape: {tfidf_tr.shape}")

# ══════════════════════════════════════════════════════════
# BUILD ALL FEATURE ARRAYS
# ══════════════════════════════════════════════════════════
print("\n[4] Building feature arrays ...")

NUM_FEAT  = NUMERIC_COLS + ORDINAL_COLS   # 5 features

scaler    = StandardScaler()
X_num_tr  = scaler.fit_transform(train_raw[NUM_FEAT].values.astype(float)).astype(np.float32)
X_num_te  = scaler.transform(test_raw[NUM_FEAT].values.astype(float)).astype(np.float32)

# stress_level as standalone (for relationship branch)
X_stress_tr = train_raw['stress_norm'].values.astype(np.float32).reshape(-1,1)
X_stress_te = test_raw['stress_norm'].values.astype(np.float32).reshape(-1,1)

# Word2Vec embeddings for categorical cols
mood_emb_tr = embed_col(train_raw['previous_day_mood'])
mood_emb_te = embed_col(test_raw['previous_day_mood'])

emo_emb_tr  = embed_col(train_raw['face_emotion_hint'])   # used in relationship branch
emo_emb_te  = embed_col(test_raw['face_emotion_hint'])

amb_emb_tr  = embed_col(train_raw['ambience_type'])
amb_emb_te  = embed_col(test_raw['ambience_type'])

rq_emb_tr   = embed_col(train_raw['reflection_quality'])  # used in relationship branch
rq_emb_te   = embed_col(test_raw['reflection_quality'])

print(f"  TF-IDF       : {tfidf_tr.shape[1]}")
print(f"  Numeric      : {X_num_tr.shape[1]}")
print(f"  Stress       : 1  (relationship branch)")
print(f"  face_emotion : {emo_emb_tr.shape[1]}  (relationship branch)")
print(f"  refl_quality : {rq_emb_tr.shape[1]}  (relationship branch)")
print(f"  mood_emb     : {mood_emb_tr.shape[1]}")
print(f"  ambience_emb : {amb_emb_tr.shape[1]}")

# ══════════════════════════════════════════════════════════
# TARGETS
# ══════════════════════════════════════════════════════════
print("\n[5] Encoding targets ...")

le_state = LabelEncoder()
y_state  = le_state.fit_transform(train_raw['emotional_state'])
y_intens = train_raw['intensity'].fillna(2).astype(float).values  # regression target

N_CLASSES = len(le_state.classes_)
print(f"  Y1 classes ({N_CLASSES}): {list(le_state.classes_)}")
print(f"  Y2 intensity range: {y_intens.min():.0f} – {y_intens.max():.0f}")

# Class weights for Y1
cw_arr  = compute_class_weight('balanced', classes=np.unique(y_state), y=y_state)
cw_dict = dict(enumerate(cw_arr))
sample_w = np.array([cw_dict[c] for c in y_state], dtype=np.float32)

# ══════════════════════════════════════════════════════════
# TRAIN / VAL SPLIT  80 / 20
# ══════════════════════════════════════════════════════════
print("\n[6] Train/Val split 80/20 ...")

tr_i, val_i = train_test_split(
    np.arange(len(train_raw)), test_size=0.2,
    random_state=42, stratify=y_state)

def sp(a): return a[tr_i], a[val_i]

Xtf_tr,  Xtf_val  = sp(tfidf_tr)
Xnum_tr, Xnum_val = sp(X_num_tr)
Xstr_tr, Xstr_val = sp(X_stress_tr)
Xemo_tr, Xemo_val = sp(emo_emb_tr)
Xrq_tr,  Xrq_val  = sp(rq_emb_tr)
Xmood_tr,Xmood_val= sp(mood_emb_tr)
Xamb_tr, Xamb_val = sp(amb_emb_tr)

ys_tr, ys_val = sp(y_state)
yi_tr, yi_val = sp(y_intens)
sw_tr         = sample_w[tr_i]

print(f"  Train:{len(tr_i)}  Val:{len(val_i)}  Test:{len(test_raw)}")

# ══════════════════════════════════════════════════════════
# DATA AUGMENTATION — Gaussian noise on numeric + tfidf
# ══════════════════════════════════════════════════════════
def augment_all(arrays_list, ys, yi, sw, n=3, noise=0.06):
    aug = [list(arrays_list)]
    ya, yia, swa = [ys], [yi], [sw]
    for _ in range(n):
        noisy = []
        for arr in arrays_list:
            n_arr = arr + np.random.normal(0, noise, arr.shape).astype(np.float32)
            noisy.append(n_arr)
        aug.append(noisy)
        ya.append(ys); yia.append(yi); swa.append(sw)
    stacked = [np.vstack([a[i] for a in aug]) for i in range(len(arrays_list))]
    return stacked, np.concatenate(ya), np.concatenate(yia), np.concatenate(swa)

train_arrays = [Xtf_tr, Xnum_tr, Xstr_tr, Xemo_tr, Xrq_tr, Xmood_tr, Xamb_tr]
aug_arrays, ys_aug, yi_aug, _ = augment_all(train_arrays, ys_tr, yi_tr, sw_tr)
Xtf_aug, Xnum_aug, Xstr_aug, Xemo_aug, Xrq_aug, Xmood_aug, Xamb_aug = aug_arrays

print(f"\n  After augmentation: {len(ys_aug)} samples (was {len(ys_tr)})")

# ══════════════════════════════════════════════════════════
# NEURAL NETWORK  — Multi-Input Multi-Output
# ══════════════════════════════════════════════════════════
print("\n[7] Building Neural Network ...")

L2 = 1e-4

def build_model():
    reg = regularizers.l2(L2)

    # ── INPUT 1: TF-IDF journal_text (300-dim) ──
    tfidf_in  = Input(shape=(TFIDF_DIM,), name='tfidf')
    x_text    = layers.Dense(128, activation='relu', kernel_regularizer=reg)(tfidf_in)
    x_text    = layers.BatchNormalization()(x_text)
    x_text    = layers.Dropout(0.2)(x_text)
    x_text    = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x_text)
    # Branch A output: 64-dim text representation

    # ── INPUT 2: stress_level (1-dim scalar) ──
    stress_in = Input(shape=(1,), name='stress')

    # ── INPUT 3: face_emotion_hint Word2Vec (8-dim) ──
    emo_in    = Input(shape=(EMBED_DIM,), name='face_emotion')

    # ── INPUT 4: reflection_quality Word2Vec (8-dim) ──
    rq_in     = Input(shape=(EMBED_DIM,), name='refl_quality')

    # ── RELATIONSHIP BRANCH ──
    # Models the relationship between stress_level,
    # face_emotion_hint, journal_text and reflection_quality
    rel_concat = layers.Concatenate(name='relationship_inputs')(
        [x_text, stress_in, emo_in, rq_in])   # 64+1+8+8 = 81
    x_rel = layers.Dense(64, activation='relu',
                          kernel_regularizer=reg, name='rel_dense1')(rel_concat)
    x_rel = layers.BatchNormalization()(x_rel)
    x_rel = layers.Dropout(0.15)(x_rel)
    x_rel = layers.Dense(32, activation='relu',
                          kernel_regularizer=reg, name='rel_dense2')(x_rel)
    # Branch B output: 32-dim relationship representation

    # ── INPUT 5: other numeric + ordinal (5-dim) ──
    num_in    = Input(shape=(len(NUM_FEAT),), name='numeric')
    x_num     = layers.Dense(32, activation='relu', kernel_regularizer=reg)(num_in)
    x_num     = layers.BatchNormalization()(x_num)
    x_num     = layers.Dropout(0.1)(x_num)
    x_num     = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x_num)
    # Branch C output: 16-dim numeric representation

    # ── INPUT 6: previous_day_mood (8-dim) ──
    mood_in   = Input(shape=(EMBED_DIM,), name='mood')

    # ── INPUT 7: ambience_type (8-dim) ──
    amb_in    = Input(shape=(EMBED_DIM,), name='ambience')

    # ── MERGE ALL BRANCHES ──
    # 64(text) + 32(relationship) + 16(numeric) + 8(mood) + 8(ambience) = 128
    # Note: stress, face_emotion, refl_quality already consumed in relationship branch
    merged = layers.Concatenate(name='all_merged')(
        [x_text, x_rel, x_num, mood_in, amb_in])

    # ── SHARED HIDDEN ──
    h = layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    h = layers.BatchNormalization()(h)
    h = layers.Dropout(0.15)(h)
    h = layers.Dense(32, activation='relu', kernel_regularizer=reg)(h)
    h = layers.BatchNormalization()(h)
    h = layers.Dropout(0.1)(h)

    # ── HEAD 1: emotional_state → Classification (Softmax) ──
    h1 = layers.Dense(32, activation='relu',
                       kernel_regularizer=reg, name='state_head')(h)
    h1 = layers.Dropout(0.1)(h1)
    out_state = layers.Dense(N_CLASSES, activation='softmax',
                              name='emotional_state')(h1)

    # ── HEAD 2: intensity → Regression (Linear) ──
    h2 = layers.Dense(16, activation='relu',
                       kernel_regularizer=reg, name='intensity_head')(h)
    out_intens = layers.Dense(1, activation='linear',
                               name='intensity')(h2)

    model = Model(
        inputs  = [tfidf_in, num_in, stress_in, emo_in, rq_in, mood_in, amb_in],
        outputs = [out_state, out_intens],
        name    = 'ArvyaX_MultiInput'
    )
    return model

model = build_model()

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=3e-4),
    loss = {
        'emotional_state': 'sparse_categorical_crossentropy',
        'intensity':        'mse',
    },
    loss_weights = {'emotional_state': 1.0, 'intensity': 0.5},
    metrics = {
        'emotional_state': ['accuracy'],
        'intensity':        ['mae'],
    }
)

model.summary()

# ══════════════════════════════════════════════════════════
# PREPARE INPUT DICTS
# ══════════════════════════════════════════════════════════
def make_inputs(tf, num, stress, emo, rq, mood, amb):
    return {
        'tfidf':        tf,
        'numeric':      num,
        'stress':       stress,
        'face_emotion': emo,
        'refl_quality': rq,
        'mood':         mood,
        'ambience':     amb,
    }

X_train_in = make_inputs(Xtf_aug, Xnum_aug, Xstr_aug,
                          Xemo_aug, Xrq_aug, Xmood_aug, Xamb_aug)
X_val_in   = make_inputs(Xtf_val, Xnum_val, Xstr_val,
                          Xemo_val, Xrq_val, Xmood_val, Xamb_val)
X_test_in  = make_inputs(tfidf_te, X_num_te, X_stress_te,
                          emo_emb_te, rq_emb_te, mood_emb_te, amb_emb_te)

Y_train    = {'emotional_state': ys_aug, 'intensity': yi_aug}
Y_val      = {'emotional_state': ys_val, 'intensity': yi_val}

# ══════════════════════════════════════════════════════════
# TRAIN
# ══════════════════════════════════════════════════════════
print("\n[8] Training ...")

callbacks = [
    EarlyStopping(monitor='val_emotional_state_accuracy', mode='max',
                  patience=30, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=12, min_lr=1e-6, verbose=0),
]


# ── Compute per-sample weights from class weights (augmented data) ──
# Keras 3.x does NOT support class_weight or sample_weight dict
# for multi-output models. Workaround: build a tf.data.Dataset
# that bundles (inputs, outputs, sample_weights) as a single tuple.
sw_aug_arr = np.array([cw_dict[c] for c in ys_aug], dtype=np.float32)

import tensorflow as _tf

def make_dataset(x_dict, y_dict, sw, batch_size, shuffle=False):
    ds = _tf.data.Dataset.from_tensor_slices((x_dict, y_dict, sw))
    if shuffle:
        ds = ds.shuffle(len(sw), seed=42)
    return ds.batch(batch_size).prefetch(_tf.data.AUTOTUNE)

BATCH = 8
train_ds = make_dataset(X_train_in, Y_train, sw_aug_arr,    BATCH, shuffle=True)
val_ds   = make_dataset(X_val_in,   Y_val,
                        np.ones(len(ys_val), dtype=np.float32), BATCH)

history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 300,
    callbacks       = callbacks,
    verbose         = 1,
)

# ══════════════════════════════════════════════════════════
# EVALUATE
# ══════════════════════════════════════════════════════════
print("\n[9] Validation results ...")

val_preds   = model.predict(X_val_in, verbose=0)
ys_pred     = np.argmax(val_preds[0], axis=1)
yi_pred_raw = val_preds[1].flatten()
yi_pred_r   = np.clip(np.round(yi_pred_raw), 1, 5).astype(int)

s_acc = accuracy_score(ys_val, ys_pred)
mae   = mean_absolute_error(yi_val, yi_pred_raw)

print(f"\n  ── Y1: Emotional State (acc={s_acc:.3f}) ──")
print(classification_report(
    le_state.inverse_transform(ys_val),
    le_state.inverse_transform(ys_pred),
    zero_division=0))

print(f"\n  ── Y2: Intensity Regression (MAE={mae:.3f}) ──")
print(f"  True   : {yi_val[:10].astype(int).tolist()}")
print(f"  Pred   : {yi_pred_r[:10].tolist()}")

# ══════════════════════════════════════════════════════════
# PLOTS
# ══════════════════════════════════════════════════════════
print("\n[10] Saving plots ...")

# Training curves
h  = history.history
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
def plot_h(ax, key, title):
    ax.plot(h.get(key,[]),        label='train', color='#534AB7', lw=2)
    ax.plot(h.get('val_'+key,[]), label='val',   color='#993C1D', lw=2, ls='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

plot_h(axes[0,0], 'emotional_state_accuracy', 'Y1 Accuracy')
plot_h(axes[0,1], 'intensity_mae',             'Y2 MAE')
plot_h(axes[1,0], 'emotional_state_loss',      'Y1 Loss')
plot_h(axes[1,1], 'loss',                      'Total Loss')
plt.tight_layout()
plt.savefig('outputs/01_training_curves.png', dpi=120, bbox_inches='tight')
plt.close()

# Confusion matrix + intensity scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(ys_val, ys_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_state.classes_,
            yticklabels=le_state.classes_, ax=axes[0])
axes[0].set_title(f'Y1 Emotional State  (acc={s_acc:.2f})')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

axes[1].scatter(yi_val, yi_pred_raw, alpha=0.6, color='#534AB7', s=60)
axes[1].plot([1,5],[1,5],'r--', lw=1.5, label='perfect')
axes[1].set_xlabel('True Intensity'); axes[1].set_ylabel('Predicted (continuous)')
axes[1].set_title(f'Y2 Intensity Regression  (MAE={mae:.2f})')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/02_results.png', dpi=120, bbox_inches='tight')
plt.close()

# Architecture diagram (text-based visual)
fig, ax = plt.subplots(figsize=(12, 7))
ax.axis('off')
arch_text = (
    "Neural Network Architecture\n\n"
    "  INPUT 1: journal_text → TF-IDF(300) → Dense(128→64)  [Branch A: Text]\n"
    "                                                  ↓\n"
    "  INPUT 2: stress_level ──────────────────────────┤\n"
    "  INPUT 3: face_emotion_hint → Word2Vec(8) ────────┤→ Dense(64→32)  [RELATIONSHIP BRANCH]\n"
    "  INPUT 4: reflection_quality → Word2Vec(8) ────────┘  (models stress×emotion×text×quality)\n\n"
    "  INPUT 5: numeric/ordinal(5) → Dense(32→16)  [Branch C: Metadata]\n"
    "  INPUT 6: previous_day_mood → Word2Vec(8)\n"
    "  INPUT 7: ambience_type → Word2Vec(8)\n\n"
    "  ALL MERGED → Dense(64→32) [Shared Hidden]\n"
    "       ├──→ Dense(32→N) + Softmax  ═══ Y1: emotional_state  [CLASSIFICATION]\n"
    "       └──→ Dense(16→1) + Linear   ═══ Y2: intensity        [REGRESSION]\n\n"
    f"  Y1 Classes: {list(le_state.classes_)}\n"
    f"  Y2 Output : continuous 1.0–5.0  (rounded for display)\n"
)
ax.text(0.02, 0.95, arch_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))
plt.tight_layout()
plt.savefig('outputs/03_architecture.png', dpi=120, bbox_inches='tight')
plt.close()

print("  Plots saved.")

# ══════════════════════════════════════════════════════════
# UNCERTAINTY
# ══════════════════════════════════════════════════════════
def get_confidence(proba):
    p = np.sort(proba)[::-1]
    conf   = float(p[0])
    margin = float(p[0]-p[1]) if len(p)>1 else 1.0
    return round(conf, 3), int(conf < 0.50 or margin < 0.15)

# ══════════════════════════════════════════════════════════
# RECOMMENDATION ENGINE
# Built from: predicted_state + intensity + stress + energy + time
# ══════════════════════════════════════════════════════════
ACTIVITY_MAP = {
    # (state, high_stress, low_energy) → activity
    ('anxious',      True,  True):  'box_breathing',
    ('anxious',      True,  False): 'box_breathing',
    ('anxious',      False, True):  'rest',
    ('anxious',      False, False): 'grounding',
    ('mixed',        True,  False): 'journaling',
    ('mixed',        False, False): 'light_planning',
    ('calm_neutral', False, False): 'light_planning',
    ('calm_neutral', True,  False): 'grounding',
    ('overwhelmed',  True,  False): 'box_breathing',
    ('overwhelmed',  False, False): 'rest',
    ('focused',      False, False): 'deep_work',
    ('focused',      True,  False): 'yoga',
    ('restless',     True,  False): 'grounding',
    ('restless',     False, False): 'movement',
}

WHEN_MAP = {
    # (state, rounded_intensity) → timing
    ('anxious',      5): 'now',
    ('anxious',      4): 'now',
    ('anxious',      3): 'within_15_min',
    ('anxious',      2): 'within_15_min',
    ('overwhelmed',  5): 'now',
    ('overwhelmed',  4): 'now',
    ('overwhelmed',  3): 'within_15_min',
    ('mixed',        4): 'within_15_min',
    ('mixed',        3): 'within_15_min',
    ('mixed',        2): 'later_today',
    ('restless',     4): 'within_15_min',
    ('restless',     3): 'within_15_min',
    ('calm_neutral', 2): 'later_today',
    ('calm_neutral', 1): 'tomorrow_morning',
    ('focused',      3): 'now',
    ('focused',      2): 'now',
    ('focused',      1): 'later_today',
}

MESSAGES = {
    'anxious':
        "Your reflection shows tension and inner noise. Before anything else, "
        "try 4 rounds of box breathing (4 counts in → hold → out → hold). "
        "Let the stress release before you plan your next step.",

    'overwhelmed':
        "You're carrying too much right now. The reflection and emotional signals "
        "both point to overload. Rest is not laziness here — it is the strategy. "
        "Pick just one small task after a break.",

    'mixed':
        "Your signals are split — some calm, some tension. Your journal shows "
        "conflicting thoughts. A short journaling or light planning session can "
        "help you clarify which direction to move.",

    'calm_neutral':
        "You're in a steady, balanced state. This is a good window for light "
        "planning or gentle focused work. Don't push too hard — maintain the calm.",

    'focused':
        "Your reflection shows mental clarity and readiness. Your face emotion "
        "and stress levels confirm it. This is your peak window — start your "
        "most important task right now.",

    'restless':
        "You seem physically or mentally unsettled. Some light movement — a short "
        "walk or stretching — can help discharge the restless energy before "
        "sitting back down to work.",
}

def decide_what(state, hs, lef):
    for k in [(state,hs,lef),(state,False,lef),(state,False,False)]:
        if k in ACTIVITY_MAP: return ACTIVITY_MAP[k]
    return 'pause'

def decide_when(state, intensity_rounded):
    for k in [(state, intensity_rounded),(state, 2)]:
        if k in WHEN_MAP: return WHEN_MAP[k]
    return 'later_today'

# ══════════════════════════════════════════════════════════
# TEST PREDICTIONS
# ══════════════════════════════════════════════════════════
print("\n[11] Generating test predictions ...")

te_preds    = model.predict(X_test_in, verbose=0)
s_probas    = te_preds[0]
yi_te_raw   = te_preds[1].flatten()
te_states   = le_state.inverse_transform(np.argmax(s_probas, axis=1))
te_intens_r = np.clip(np.round(yi_te_raw), 1, 5).astype(int)

records = []
for i in range(len(test_raw)):
    state     = te_states[i]
    intensity = int(te_intens_r[i])
    conf, unc = get_confidence(s_probas[i])

    stress_val = float(test_raw.iloc[i]['stress_level'])
    energy_val = float(test_raw.iloc[i]['energy_level'])
    hs  = stress_val >= 4
    lef = energy_val <= 2

    what    = decide_what(state, hs, lef)
    when    = decide_when(state, intensity)
    message = MESSAGES.get(state, "Take a moment to pause and check in.")

    records.append({
        'id':                  test_raw.iloc[i]['id'],
        'predicted_state':     state,
        'predicted_intensity': intensity,
        'intensity_raw':       round(float(yi_te_raw[i]), 2),
        'confidence':          conf,
        'uncertain_flag':      unc,
        'what_to_do':          what,
        'when_to_do':          when,
        'supportive_message':  message,
    })

pred_df = pd.DataFrame(records)

print("\n  ── Test Predictions ──")
print(pred_df[['id','predicted_state','predicted_intensity',
               'confidence','uncertain_flag','what_to_do','when_to_do']].to_string(index=False))

print("\n  ── Supportive Messages ──")
for _, r in pred_df.iterrows():
    print(f"\n  ID {r['id']} [{r['predicted_state'].upper()}, intensity={r['predicted_intensity']}]")
    print(f"  {r['supportive_message']}")

# Save
pred_df[['id','predicted_state','predicted_intensity',
         'confidence','uncertain_flag','what_to_do','when_to_do']].to_csv(
    'outputs/predictions.csv', index=False)
pred_df.to_csv('outputs/predictions_full.csv', index=False)



  ArvyaX — NN  |  TF-IDF + Embeddings + Relationship Modeling

  Train: 1200  |  Test: 120

[1] Preprocessing ...
  Numeric medians:
    duration_min    = 15.00
    sleep_hours     = 6.00
    energy_level    = 3.00
    stress_level    = 3.00

  Classes after calm+neutral merge:
    calm_neutral    417 (34.8%)
    restless        209 (17.4%)
    focused         193 (16.1%)
    mixed           191 (15.9%)
    overwhelmed     190 (15.8%)

[2] Training Word2Vec ...
  Vocabulary size: 517

[3] TF-IDF on journal_text ...
  TF-IDF shape: (1200, 300)

[4] Building feature arrays ...
  TF-IDF       : 300
  Numeric      : 5
  Stress       : 1  (relationship branch)
  face_emotion : 8  (relationship branch)
  refl_quality : 8  (relationship branch)
  mood_emb     : 8
  ambience_emb : 8

[5] Encoding targets ...
  Y1 classes (5): ['calm_neutral', 'focused', 'mixed', 'overwhelmed', 'restless']
  Y2 intensity range: 1 – 5

[6] Train/Val split 80/20 ...
  Train:960  Val:240  Test:120

  After augment

Model: "ArvyaX_MultiInput"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tfidf (InputLayer)  │ (None, 300)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 128)       │     38,528 │ tfidf[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_24[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_24          │ (None, 128)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 64)        │      8,256 │ dropout_24[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stress (InputLayer) │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ face_emotion        │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ refl_quality        │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relationship_inputs │ (None, 81)        │          0 │ dense_25[0][0],   │
│ (Concatenate)       │                   │            │ stress[0][0],     │
│                     │                   │            │ face_emotion[0][… │
│                     │                   │            │ refl_quality[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric             │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rel_dense1 (Dense)  │ (None, 64)        │      5,248 │ relationship_inp… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 32)        │        192 │ numeric[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ rel_dense1[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense_26[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_25          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_26          │ (None, 32)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rel_dense2 (Dense)  │ (None, 32)        │      2,080 │ dropout_25[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 16)        │        528 │ dropout_26[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mood (InputLayer)   │ (None, 8)         │          0 │ -               

 Total params: 68,214 (266.46 KB)

 Trainable params: 67,574 (263.96 KB)

 Non-trainable params: 640 (2.50 KB)


[8] Training ...
Epoch 1/300
480/480 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - emotional_state_accuracy: 0.1844 - emotional_state_loss: 1.8368 - intensity_loss: 8.2081 - intensity_mae: 2.4332 - loss: 5.9999 - val_emotional_state_accuracy: 0.1708 - val_emotional_state_loss: 1.7015 - val_intensity_loss: 4.1226 - val_intensity_mae: 1.6545 - val_loss: 3.8219 - learning_rate: 3.0000e-04
Epoch 2/300
480/480 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - emotional_state_accuracy: 0.2005 - emotional_state_loss: 1.7174 - intensity_loss: 3.1905 - intensity_mae: 1.4673 - loss: 3.3718 - val_emotional_state_accuracy: 0.1958 - val_emotional_state_loss: 1.6778 - val_intensity_loss: 2.7535 - val_intensity_mae: 1.3354 - val_loss: 3.1137 - learning_rate: 3.0000e-04
Epoch 3/300
480/480 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - emotional_state_accuracy: 0.2177 - emotional_state_loss: 1.6818 - intensity_loss: 2.6236 - intensity_mae: 1.3530 - loss: 3.0527 - val_emotional_state_accuracy: 0.2208 - val_emotional_state_loss: 1.6505 -

KeyError: 'id'